# Text Classifier with Trainable Word Embeddings

Binary classification of news headlines: **Sports (0)** vs **Technology (1)**

**Architecture:** `Embedding → AvgPool → Linear(64, Sigmoid) → Linear(2)`  
**Goal:** Train the model and observe how word embeddings evolve — related words should converge, unrelated words should diverge.

## 1. Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np

## 2. Dataset

In [2]:
train_texts = [
    # Sports (100 samples) — Label 0
    "Lakers win championship after amazing comeback",
    "Football team scores three goals in final match",
    "Tennis star wins grand slam tournament",
    "Olympic athlete breaks world record",
    "Basketball game ends in dramatic overtime",
    "Soccer team advances to finals",
    "Runner wins marathon in record time",
    "Baseball team clinches playoff spot",
    "Swimmer wins gold medal at Olympics",
    "Hockey team defeats rival in shootout",
    "Gymnast performs perfect routine at competition",
    "Boxer wins title fight in final round",
    "Cycling champion wins Tour de France stage",
    "Golfer sinks incredible putt to win tournament",
    "Figure skater lands difficult jump perfectly",
    "Wrestling champion defends title successfully",
    "Cricket team wins test match series",
    "Rugby team scores last minute try",
    "Volleyball team wins championship match",
    "Badminton player wins international tournament",
    "Team celebrates victory after tough game",
    "Players train hard for upcoming season",
    "Coach announces new strategy for playoffs",
    "Stadium packed with fans for final game",
    "Athletes compete in regional championship",
    "Team captain leads squad to victory",
    "Young player shows great potential",
    "Fans celebrate team winning streak",
    "Manager signs new contract with club",
    "Training camp begins for new season",
    "Team wins away game against rivals",
    "Player receives award for performance",
    "Championship game draws huge crowd",
    "Team prepares for important match",
    "Coach praises players after victory",
    "Athletes set new records at event",
    "Team qualifies for next round",
    "Player scores hat trick in match",
    "Championship trophy awarded to winners",
    "Team celebrates historic achievement",
    "Players excited for tournament start",
    "Coach announces starting lineup today",
    "Team practices before big game",
    "Victory parade celebrates championship win",
    "Players sign autographs for fans",
    "Team mascot entertains crowd at game",
    "Championship banner raised at stadium",
    "Team dominates in playoff series",
    "Player injury sidelines star athlete",
    "Team rebuilds roster for season",
    "Coach implements new training program",
    "Players bond during team retreat",
    "Stadium renovations completed for season",
    "Team announces ticket prices for games",
    "Player traded to rival team",
    "Coach resigns after disappointing season",
    "Team practices penalty kicks for game",
    "Championship ring ceremony held today",
    "Team wins tournament in overtime",
    "Player breaks scoring record in game",
    "Coach gives motivational speech to team",
    "Team celebrates winning season finale",
    "Athletes train for upcoming competition",
    "Player signs endorsement deal today",
    "Team unveils new uniform design",
    "Championship parade draws massive crowd",
    "Coach analyzes game film with players",
    "Team defeats defending champions convincingly",
    "Player receives sportsmanship award today",
    "Team holds charity event for fans",
    "Athletes prepare for championship match",
    "Coach praises team effort after win",
    "Team advances in tournament bracket",
    "Player scores winning goal in match",
    "Championship celebrations continue all week",
    "Team announces preseason schedule today",
    "Players work hard during practice",
    "Coach happy with team performance",
    "Team wins decisive game at home",
    "Athletes excited for season opener",
    "Player makes incredible save in game",
    "Team rallies from behind for victory",
    "Coach confident before important match",
    "Championship run inspires young athletes",
    "Team prepares strategy for rivals",
    "Player demonstrates leadership on field",
    "Team celebrates milestone victory today",
    "Coach reviews tactics with players",
    "Athletes compete in regional finals",
    "Player achieves personal best in event",
    "Team practices formations for game",
    "Championship trophy displayed at stadium",
    "Coach motivates team before playoffs",
    "Team wins thrilling match in finale",
    "Player earns spot on national team",
    "Athletes train for international competition",
    "Team holds press conference today",
    # Technology (100 samples) — Label 1
    "New smartphone features advanced AI technology",
    "Tech company releases latest software update",
    "Scientists develop breakthrough quantum computer",
    "Artificial intelligence system improves healthcare",
    "New app helps users learn programming",
    "Electric vehicle company announces new model",
    "Researchers create faster internet connection",
    "Social media platform adds new features",
    "Cloud computing service expands globally",
    "Cybersecurity system protects against attacks",
    "Virtual reality headset launches next month",
    "Machine learning algorithm solves complex problem",
    "New programming language released by developers",
    "Tech startup raises millions in funding",
    "5G network expands to more cities",
    "Robotics company builds autonomous system",
    "Data center uses renewable energy",
    "Blockchain technology improves security",
    "New laptop features powerful processor",
    "Software update fixes major bugs",
    "Company develops innovative tech solution",
    "Algorithm improves search accuracy online",
    "Digital platform streamlines business operations",
    "Innovation drives tech industry forward",
    "Startup creates app for education",
    "Computer chip breakthrough announced today",
    "Technology advances medical research capabilities",
    "New software helps developers code faster",
    "Tech firm invests in AI research",
    "Digital transformation changes business landscape",
    "Scientists program robot for tasks",
    "Tech company expands into new markets",
    "Innovation lab opens in Silicon Valley",
    "New technology revolutionizes communication industry",
    "Software engineers develop better tools",
    "Tech conference showcases latest innovations",
    "Startup creates platform for collaboration",
    "Computer vision system recognizes objects accurately",
    "Technology enables remote work solutions",
    "Digital security measures protect data",
    "Algorithm optimizes supply chain operations",
    "Tech giant announces quarterly earnings today",
    "Innovation accelerates in artificial intelligence",
    "Software update enhances user experience",
    "Tech startup disrupts traditional industry",
    "Computer scientists breakthrough in research",
    "Digital platform connects developers worldwide",
    "Technology transforms healthcare delivery system",
    "New app simplifies complex tasks",
    "Innovation drives efficiency in business",
    "Tech company partners with university",
    "Software development tools improve productivity",
    "Algorithm processes data faster now",
    "Technology enables smart home systems",
    "Digital innovation changes customer experience",
    "Tech firm releases open source software",
    "Computer program automates repetitive work",
    "Innovation creates new tech opportunities",
    "Software engineers collaborate on project",
    "Technology advances renewable energy solutions",
    "Digital tools help students learn",
    "Tech startup solves infrastructure problem",
    "Algorithm improves recommendation system accuracy",
    "Innovation drives semiconductor industry growth",
    "Software platform integrates multiple services",
    "Technology enables precision medicine approach",
    "Digital assistant becomes more intelligent",
    "Tech company invests in quantum research",
    "Computer network expands bandwidth capacity",
    "Innovation transforms financial services industry",
    "Software developers build mobile applications",
    "Technology improves agricultural productivity significantly",
    "Digital marketplace connects buyers sellers",
    "Tech firm announces new partnership",
    "Algorithm detects patterns in data",
    "Innovation accelerates autonomous vehicle development",
    "Software update adds security features",
    "Technology enables virtual collaboration tools",
    "Digital platform supports remote learning",
    "Tech startup creates innovative solution",
    "Computer system processes information quickly",
    "Innovation drives biotechnology research forward",
    "Software engineers optimize application performance",
    "Technology transforms manufacturing processes today",
    "Digital tools enhance creative workflows",
    "Tech company expands research division",
    "Algorithm improves translation accuracy significantly",
    "Innovation creates sustainable technology solutions",
    "Software platform enables data analysis",
    "Technology advances space exploration capabilities",
    "Digital infrastructure supports cloud services",
    "Tech firm develops edge computing solution",
    "Computer scientists research neural networks",
    "Innovation drives telecommunications industry growth",
    "Software development becomes more accessible",
    "Technology enables personalized learning experiences",
    "Digital security protects online transactions",
]

train_labels = [0] * 100 + [1] * 100

test_texts = [
    # Sports (25 samples) — Label 0
    "Team wins game in final seconds",
    "Player scores amazing goal today",
    "Coach proud of team performance",
    "Athletes compete in championship finals",
    "Team defeats rivals in playoff",
    "Player breaks record in tournament",
    "Championship game ends in victory",
    "Team prepares for important match",
    "Athletes train for upcoming season",
    "Player receives award for excellence",
    "Team celebrates winning streak today",
    "Coach announces strategy for game",
    "Championship trophy presented to team",
    "Player scores in overtime win",
    "Team advances to next round",
    "Athletes perform well at competition",
    "Coach motivates players before match",
    "Team wins decisive playoff game",
    "Player demonstrates skill on field",
    "Championship victory celebrated by fans",
    "Team dominates in tournament play",
    "Athletes excited for season start",
    "Player achieves milestone in career",
    "Team practices before big match",
    "Championship parade honors winning team",
    # Technology (25 samples) — Label 1
    "Software company releases new product",
    "Algorithm improves system performance significantly",
    "Tech startup develops innovative platform",
    "Digital tools enhance productivity today",
    "Innovation drives technology sector forward",
    "Computer program solves difficult problem",
    "Technology transforms business operations completely",
    "Software engineers create better solutions",
    "Tech firm announces breakthrough research",
    "Digital platform connects users globally",
    "Algorithm processes information efficiently now",
    "Innovation accelerates in tech industry",
    "Software update improves functionality greatly",
    "Technology enables new capabilities today",
    "Tech company invests in development",
    "Digital system automates complex tasks",
    "Innovation creates technology opportunities now",
    "Software platform integrates services seamlessly",
    "Technology advances research capabilities significantly",
    "Tech startup solves industry challenge",
    "Algorithm optimizes operations effectively today",
    "Innovation drives digital transformation forward",
    "Software developers build applications efficiently",
    "Technology improves user experience greatly",
    "Digital innovation changes industry landscape",
]

test_labels = [0] * 25 + [1] * 25

print(f"Train: {len(train_texts)} samples  |  Test: {len(test_texts)} samples")

Train: 194 samples  |  Test: 50 samples


## 3. Vocabulary & Encoding

In [3]:
def build_vocab(texts):
    """Build word → index mapping from training corpus."""
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for text in texts:
        for word in text.lower().split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab


def encode(text, vocab):
    """Convert text to list of token indices."""
    return [vocab.get(w, vocab["<UNK>"]) for w in text.lower().split()]


vocab = build_vocab(train_texts)
print(f"Vocabulary size: {len(vocab)}")

Vocabulary size: 477


## 4. Dataset & DataLoader

In [4]:
class HeadlineDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.samples = [
            (torch.tensor(encode(t, vocab), dtype=torch.long), torch.tensor(l))
            for t, l in zip(texts, labels)
        ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def collate_fn(batch):
    """Pad variable-length sequences within a batch."""
    seqs, labels = zip(*batch)
    padded = nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=0)
    return padded, torch.stack(labels)


train_ds     = HeadlineDataset(train_texts, train_labels, vocab)
test_ds      = HeadlineDataset(test_texts,  test_labels,  vocab)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  collate_fn=collate_fn)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, collate_fn=collate_fn)

print(f"Batches — Train: {len(train_loader)}  |  Test: {len(test_loader)}")

Batches — Train: 7  |  Test: 2


## 5. Model

In [5]:
class EmbeddingClassifier(nn.Module):
    """
    Architecture
    ------------
    1. Embedding   : vocab_size × embedding_dim  (trainable)
    2. AvgPool     : mean over non-padding tokens → (batch, embedding_dim)
    3. Hidden      : Linear(embedding_dim → 64) + Sigmoid
    4. Output      : Linear(64 → 2)  — CrossEntropyLoss handles softmax
    """

    def __init__(self, vocab_size, embedding_dim=50, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.hidden    = nn.Linear(embedding_dim, hidden_dim)
        self.output    = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        emb    = self.embedding(x)                                          # (batch, seq, emb)
        mask   = (x != 0).unsqueeze(-1).float()
        pooled = (emb * mask).sum(1) / mask.sum(1).clamp(min=1e-9)         # masked avg
        hidden = torch.sigmoid(self.hidden(pooled))
        return self.output(hidden)


model = EmbeddingClassifier(len(vocab))
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

EmbeddingClassifier(
  (embedding): Embedding(477, 50, padding_idx=0)
  (hidden): Linear(in_features=50, out_features=64, bias=True)
  (output): Linear(in_features=64, out_features=2, bias=True)
)

Total parameters: 27,244


## 6. Embedding Analysis Utilities

In [6]:
def get_embedding(model, vocab, word):
    """Return embedding vector for a single word."""
    idx = vocab.get(word, vocab["<UNK>"])
    return model.embedding.weight[idx].detach()


def cosine_sim(a, b):
    """Cosine similarity between two 1-D tensors."""
    return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()


def snapshot(model, vocab, words):
    """Save current embedding vectors for a list of words."""
    return {w: get_embedding(model, vocab, w).clone() for w in words}


# Words to track
tracked_words   = ["team", "game", "algorithm"]
related_pairs   = [("team", "game")]           # both sports → should converge
unrelated_pairs = [("team", "algorithm")]      # sports vs tech → should diverge
all_pairs       = related_pairs + unrelated_pairs

## 7. Training Loop

In [7]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0.0, 0
    for x, y in loader:
        optimizer.zero_grad()
        logits = model(x)
        loss   = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct    += (logits.argmax(1) == y).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n


def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in loader:
            correct += (model(x).argmax(1) == y).sum().item()
    return correct / len(loader.dataset)


# Hyperparameters
EPOCHS = 20
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
criterion = nn.CrossEntropyLoss()

# Snapshot at epoch 0 (before training)
emb_e0 = snapshot(model, vocab, tracked_words)

print(f"{'Epoch':>6}  {'Train Loss':>10}  {'Train Acc':>9}  {'Test Acc':>8}")
print("-" * 42)

for epoch in range(1, EPOCHS + 1):
    loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    test_acc        = evaluate(model, test_loader)
    if epoch % 5 == 0 or epoch == 1:
        print(f"{epoch:>6}  {loss:>10.4f}  {train_acc:>9.3f}  {test_acc:>8.3f}")

# Snapshot at epoch 20 (after training)
emb_e20 = snapshot(model, vocab, tracked_words)

 Epoch  Train Loss  Train Acc  Test Acc
------------------------------------------
     1      0.7285      0.505     0.580
     5      0.6921      0.495     0.500
    10      0.7411      0.485     0.640
    15      0.6979      0.649     0.780
    20      0.6637      0.665     0.880


## 8. Embedding Analysis

### 8.1 Pairwise Cosine Similarity — Epoch 0 vs Epoch 20

In [8]:
print(f"{'Pair':<25} {'Epoch 0':>10} {'Epoch 20':>10} {'Change':>10}")
print("-" * 58)
for w1, w2 in all_pairs:
    sim0  = cosine_sim(emb_e0[w1],  emb_e0[w2])
    sim20 = cosine_sim(emb_e20[w1], emb_e20[w2])
    print(f"{w1 + ' / ' + w2:<25} {sim0:>10.4f} {sim20:>10.4f} {sim20 - sim0:>+10.4f}")

Pair                         Epoch 0   Epoch 20     Change
----------------------------------------------------------
team / game                   0.0518     0.0541    +0.0024
team / algorithm             -0.0693    -0.0705    -0.0012


### 8.2 L2 Embedding Shift per Word

In [9]:
print(f"{'Word':<15} {'L2 Shift (epoch 0 → 20)':>24}")
print("-" * 40)
for w in tracked_words:
    shift = (emb_e20[w] - emb_e0[w]).norm().item()
    print(f"{w:<15} {shift:>24.4f}")

Word             L2 Shift (epoch 0 → 20)
----------------------------------------
team                              0.0576
game                              0.0146
algorithm                         0.0107
